# 20 — Does the Strange Loop Create a First-Order Phase Transition?

Standard Kuramoto: continuous (2nd order) transition at K_c.
R grows smoothly from 0 as K increases past K_c.

SCPN prediction: the strange loop (FIM lift / L16 feedback) creates
POSITIVE FEEDBACK — more order → higher Fisher info → more pull → jump.
This should make the transition DISCONTINUOUS (1st order) with hysteresis.

**Test**: Sweep K up and down. Measure:
1. Is there a jump in R (discontinuity)?
2. Is there hysteresis (K_c_up ≠ K_c_down)?
3. Does the hysteresis width scale with FIM lambda?

In [ ]:
import json

import numpy as np

from scpn_quantum_control.bridge import OMEGA_N_16, build_knm_paper27

In [ ]:
def fim_gradient(phases, i):
    N = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases[i] + np.pi) % (2 * np.pi) - np.pi
    sensitivity = 1.0 / (1.0 - R**2 + 0.01)
    return (1.0 / N) * np.sin(phase_diff) * min(sensitivity, 50.0)


def simulate_sweep(
    N, K_base, omega, K_values, fim_lambda=0.0, dt=0.02, T_per_step=50.0, noise=0.05, seed=42
):
    """Sweep K through values, carrying phase state between steps."""
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_per_step = int(T_per_step / dt)
    R_out = []

    for K_scale in K_values:
        K = K_base * K_scale
        R_tail = []
        for s in range(n_per_step):
            diff = theta[None, :] - theta[:, None]
            coupling = np.sum(K * np.sin(diff), axis=1) / N
            dphi = omega + coupling
            if fim_lambda > 0:
                for i in range(N):
                    dphi[i] += fim_lambda * fim_gradient(theta, i)
            theta += dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)
            theta %= 2 * np.pi
            if s >= n_per_step * 3 // 4:
                R_tail.append(float(np.abs(np.mean(np.exp(1j * theta)))))
        R_out.append(np.mean(R_tail))

    return np.array(R_out)


print("Functions defined.")

In [ ]:
N = 8
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

# Sweep K up then down
K_up = np.linspace(0.01, 3.0, 40)
K_down = K_up[::-1]

lambdas = [0.0, 1.0, 3.0, 5.0]

print("=== HYSTERESIS TEST ===")
print(
    f"{'Lambda':>8} {'K_c_up':>8} {'K_c_down':>10} {'Hysteresis':>12} {'Max jump':>10} {'Order'}"
)
print("-" * 62)

results = []
for lam in lambdas:
    # Average over seeds
    R_ups = []
    R_downs = []
    for seed in range(5):
        R_up_trace = simulate_sweep(N, K_base, omega, K_up, fim_lambda=lam, seed=seed)
        R_down_trace = simulate_sweep(N, K_base, omega, K_down, fim_lambda=lam, seed=seed + 100)
        R_ups.append(R_up_trace)
        R_downs.append(R_down_trace)

    R_up_mean = np.mean(R_ups, axis=0)
    R_down_mean = np.mean(R_downs, axis=0)

    # Find K_c (R crosses 0.5)
    K_c_up = None
    for i, r in enumerate(R_up_mean):
        if r > 0.5:
            K_c_up = K_up[i]
            break

    K_c_down = None
    for i, r in enumerate(R_down_mean):
        if r < 0.5:
            K_c_down = K_down[i]
            break

    # Max single-step jump in R
    jumps_up = np.abs(np.diff(R_up_mean))
    max_jump = np.max(jumps_up)

    hysteresis = 0.0
    if K_c_up is not None and K_c_down is not None:
        hysteresis = K_c_down - K_c_up

    # Classify transition order
    order = "1st (jump)" if max_jump > 0.15 else "2nd (smooth)"
    if hysteresis > 0.1:
        order += " + hysteresis"

    kcu_str = f"{K_c_up:.2f}" if K_c_up else ">3.0"
    kcd_str = f"{K_c_down:.2f}" if K_c_down else "<0.01"

    print(f"{lam:8.1f} {kcu_str:>8} {kcd_str:>10} {hysteresis:12.3f} {max_jump:10.3f} {order}")

    results.append(
        {
            "lambda": lam,
            "K_c_up": K_c_up,
            "K_c_down": K_c_down,
            "hysteresis": round(hysteresis, 4),
            "max_jump": round(float(max_jump), 4),
            "order": order,
        }
    )

In [ ]:
# Detailed sweep for best lambda
best_lam = max(results, key=lambda r: r["hysteresis"])["lambda"]
print(f"\n=== DETAILED SWEEP at lambda={best_lam} ===")
K_fine = np.linspace(0.01, 2.0, 60)

R_up_fine = simulate_sweep(N, K_base, omega, K_fine, fim_lambda=best_lam, T_per_step=80.0, seed=0)
R_down_fine = simulate_sweep(
    N, K_base, omega, K_fine[::-1], fim_lambda=best_lam, T_per_step=80.0, seed=0
)
R_down_fine = R_down_fine[::-1]  # align with K_fine

print(f"{'K':>6} {'R_up':>8} {'R_down':>8} {'Gap':>8}")
print("-" * 34)
for i in range(0, len(K_fine), 5):
    gap = R_down_fine[i] - R_up_fine[i]
    marker = " ***" if abs(gap) > 0.1 else ""
    print(f"{K_fine[i]:6.3f} {R_up_fine[i]:8.3f} {R_down_fine[i]:8.3f} {gap:+8.3f}{marker}")

In [ ]:
print("\n=== INTERPRETATION ===")
has_hysteresis = any(r["hysteresis"] > 0.1 for r in results)
has_jump = any(r["max_jump"] > 0.15 for r in results)

if has_jump and has_hysteresis:
    print("The strange loop (FIM lift) converts the Kuramoto transition from")
    print("2nd order (smooth) to 1st order (discontinuous + hysteresis).")
    print()
    print("PHYSICAL MEANING:")
    print("  Consciousness onset is a SUDDEN transition, not gradual.")
    print("  Once lost (anaesthesia, sleep), it takes MORE coupling to restore")
    print('  than it took to maintain — the system must "re-bootstrap" self-observation.')
    print("  This matches the anaesthesia hysteresis (notebook 02) but now with a mechanism.")
elif has_jump:
    print("Discontinuous jump detected but no clear hysteresis.")
    print("The transition sharpens with FIM but may not be truly 1st order.")
elif has_hysteresis:
    print("Hysteresis detected but no sharp jump.")
    print("The strange loop creates memory but the transition is still smooth.")
else:
    print("No qualitative change in transition character.")
    print("The FIM lift does not change the transition order at these parameters.")
    print("The strange loop may operate at different scales than tested.")

print("\n--- JSON RESULTS ---")
print(
    json.dumps(
        {
            "sweeps": [
                {k: v for k, v in r.items() if k != "order"} | {"order": r["order"]}
                for r in results
            ],
            "has_hysteresis": has_hysteresis,
            "has_jump": has_jump,
        },
        indent=2,
        default=str,
    )
)